# **DATA PIPELINE**
### **Streaming Table**

In [ ]:
# Environment configuration — replace the placeholder in the Databricks widget.
dbutils.widgets.text("storage_account", "<YOUR_STORAGE_ACCOUNT>")
storage_account = dbutils.widgets.get("storage_account")
if not storage_account or storage_account.startswith("<"):
    raise ValueError("Set the storage_account widget before running.")


In [ ]:
import dlt
from pyspark.sql.functions import *

In [ ]:
# Creating expectations (rules) to follow for pipeline
rules = {
    "rule1" : "product_id IS NOT NULL",
    "rule2" : "product_name IS NOT NULL",
}

In [ ]:
@dlt.table()

@dlt.expect_all_or_drop(rules)

def DimProducts_stage():
  return (spark.readStream
          .option("skipChangeCommits", "true")  # Skip overwrite commits
          .table("databricks_etl_catalog.silver.products_silver"))

### **Streaming View**

In [ ]:
import dlt
# Creating empty streaming table

dlt.create_streaming_table("DimProducts")

In [ ]:
@dlt.view()

def DimProducts_view():
    df = spark.readStream.table("Live.DimProducts_stage")
    return df


### **DimProducts as SCD Type 2 with the help of DLT** 

In [ ]:
dlt.apply_changes(
    target = "DimProducts",
    source = "Live.DimProducts_view",
    keys = ["product_id"],
    sequence_by = "product_id",
    stored_as_scd_type = 2
)
